In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# --- Paths & Hyperparams ---
CSV_PATH        = "/home/iambrink/NOH_Thyroid_Cancer_Data/CSV-files/Thyroid_Cancer_TAN&NOH_file.csv"
BASE_IMAGE_PATH = "/home/iambrink/NOH_Thyroid_Cancer_Data/superdata/"

# Replace the Virchow2 model with MahmoodLab/UNI
MODEL_NAME  = "hf-hub:MahmoodLab/UNI"
NUM_CLASSES = 2
BATCH_SIZE  = 8
NUM_EPOCHS  = 2000
LR          = 5e-3
WD          = 1e-4
NUM_WORKERS = 8
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load & Split DataFrame ---
df = pd.read_csv(CSV_PATH).dropna(subset=["Surgery diagnosis in number"])
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Surgery diagnosis in number"]
)

# --- Custom Dataset for Thyroid Slides ---
class ThyroidDataset(Dataset):
    def __init__(self, df, base_path, transform=None):
        self.df = df.reset_index(drop=True)
        self.base = base_path
        self.tf = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Construct full path, ensure slashes are correct
        img_path = os.path.join(self.base, row["image_path"].replace("\\", "/"))
        img = Image.open(img_path).convert("RGB")
        label = int(row["Surgery diagnosis in number"])  # 0 or 1
        if self.tf:
            img = self.tf(img)
        return img, torch.tensor(label, dtype=torch.long)

# --- Data Transforms & DataLoaders ---
# Resolve the input size / normalization from the UNI model’s config
config         = resolve_data_config({}, model=MODEL_NAME)
train_transform = create_transform(**config, is_training=True)
val_transform   = create_transform(**config, is_training=False)

train_ds = ThyroidDataset(train_df, BASE_IMAGE_PATH, train_transform)
val_ds   = ThyroidDataset(val_df,   BASE_IMAGE_PATH, val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# --- Build the Model & Freeze Backbone ---
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)


# Freeze all parameters except the classifier head
for param in model.parameters():
    param.requires_grad = False

for param in model.get_classifier().parameters():
    param.requires_grad = True

# --- Loss, Optimizer, Scheduler, and Mixed Precision ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WD
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.cuda.amp.GradScaler()
torch.backends.cudnn.benchmark = True

best_val_acc = 0.0

# --- Training & Validation Loop ---
for epoch in range(1, NUM_EPOCHS + 1):
    # — Train —
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            # For UNI, depending on its architecture, it might return patch‐level logits or a single vector.
            # Here we assume it returns patch logits (like Virchow2), so we average over patches:
            outputs = model(imgs)                  # [B, num_patches, 2]  (or [B, 2] if UNI is already a classifier)
            if outputs.ndim == 3:
                logits = outputs.mean(dim=1)       # [B, 2]
            else:
                logits = outputs                  # [B, 2]
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)

    scheduler.step()
    avg_train_loss = running_loss / len(train_ds)

    # — Validate —
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                if outputs.ndim == 3:
                    logits = outputs.mean(dim=1)
                else:
                    logits = outputs
                loss   = criterion(logits, labels)

            val_loss += loss.item() * imgs.size(0)
            preds    = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_val_loss = val_loss / len(val_ds)
    val_acc      = correct / total

    print(
        f"Epoch {epoch:2d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss:   {avg_val_loss:.4f} | "
        f"Val Acc:    {val_acc:.4f}"
    )

    # Save the best‐performing checkpoint (by validation accuracy)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = "/home/iambrink/NOH_Thyroid_Cancer_Data/MODELS/best_uni_model.pth"
        torch.save(model.state_dict(), save_path)
        print(f"→ Saved new best UNI model (Acc: {best_val_acc:.4f})")


/home/iambrink/miniconda3/envs/tf-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Error(s) in loading state_dict for VisionTransformer:
	size mismatch for blocks.0.mlp.fc2.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 2048]).
	size mismatch for blocks.1.mlp.fc2.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 2048]).
	size mismatch for blocks.2.mlp.fc2.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 2048]).
	size mismatch for blocks.3.mlp.fc2.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 2048]).
	size mismatch for blocks.4.mlp.fc2.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 2048]).
	size mismatch for blocks.5.mlp.fc2.weight: copying a param with shape torch.Size([1024